<a href="https://colab.research.google.com/github/Rut092/AI-Journey-Practice/blob/main/K_means(Handling_mixed_data%2CMCA%2CMini_Batch).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Handling the Mixed Data

In [3]:
!pip install kmodes

import pandas as pd
from kmodes.kprototypes import KPrototypes


In [5]:

df = pd.DataFrame({
    'Age': [25, 34, 22, 45, 31],          # Numerical (Index 0)
    'Income': [55000, 70000, 48000, 90000, 68000], # Numerical (Index 1)
    'Device': ['Mobile', 'Desktop', 'Mobile', 'Tablet', 'Desktop'] # Categorical (Index 2)
})


kproto = KPrototypes(n_clusters=2, max_iter=10)
clusters = kproto.fit_predict(df, categorical=[2])

print(kproto.cluster_centroids_)
df['clusters'] = clusters
print(df)

[['23.5' '51500.0' 'Mobile']
 ['36.666666666666664' '76000.0' 'Desktop']]
   Age  Income   Device  clusters
0   25   55000   Mobile         0
1   34   70000  Desktop         1
2   22   48000   Mobile         0
3   45   90000   Tablet         1
4   31   68000  Desktop         1


## MCA + K-Means

In [7]:
#we use this when we have massive amounts of categorical data and
#want to shrink it down into continuous numbers so standard
#Scikit-Learn tools(like K-Means or DBSCAN) can understand it.

!pip install prince
import prince
from sklearn.cluster import KMeans


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.3/197.3 kB 5.7 MB/s eta 0:00:00


In [9]:
df_cats = pd.DataFrame({
    'City': ['NY', 'LA', 'NY', 'SF', 'LA'],
    'Browser': ['Chrome', 'Safari', 'Chrome', 'Chrome', 'Firefox']
})

# 2. Run MCA (Multiple Correspondence Analysis)
# This squashes the categories down into 2 continuous numerical columns
mca = prince.MCA(n_components=2, random_state=42)
mca_continuous_data = mca.fit_transform(df_cats)
print(mca_continuous_data)
print()
# 3. Run K-Means
kmeans = KMeans(n_clusters=2, random_state=42)
clusters = kmeans.fit_predict(mca_continuous_data)

df_cats['clusters'] = clusters
print(df_cats)

          0             1
0 -0.816497 -6.454972e-01
1  1.224745 -4.429052e-16
2 -0.816497 -6.454972e-01
3 -0.816497  1.290994e+00
4  1.224745 -5.699572e-16

  City  Browser  clusters
0   NY   Chrome         1
1   LA   Safari         0
2   NY   Chrome         1
3   SF   Chrome         1
4   LA  Firefox         0


## Mini-Batch K-Means (Handling Millions of Rows)

In [10]:
from sklearn.datasets import make_blobs
from sklearn.cluster import MiniBatchKMeans
import time

# 1. Simulate a MASSIVE dataset (1 Million rows)
# In a real-world scenario with millions of data points, creating such a large dataset in memory
# can be challenging. MiniBatchKMeans is particularly useful here as it processes data in small batches.
X_huge, _ = make_blobs(n_samples=1000000, centers=5, random_state=42)

# 2. Initialize Mini-Batch K-Means
# batch_size=1024 means it will only look at 1,024 random rows per step
mb_kmeans = MiniBatchKMeans(
    n_clusters=5, # In a real scenario, the optimal number of clusters (k) would typically be determined
                  # using methods like the Elbow method or Silhouette score, or based on domain knowledge.
                  # Here, we know it's 5 because we simulated the data with 5 centers.
    batch_size=1024,
    random_state=42,
    n_init="auto" # "auto" automatically selects the number of initializations to be performed.
                  # It's a robust default that chooses the best initialization based on different centroid seeds.
)

# 3. Time the execution
start_time = time.time()
clusters = mb_kmeans.fit_predict(X_huge)
end_time = time.time()

print(f"✅ Clustered 1,000,000 rows in just {end_time - start_time:.2f} seconds!")

✅ Clustered 1,000,000 rows in just 0.16 seconds!
